# DefinitionBot — Spec-definition workflow

Demonstrates the four-step flow that DefinitionBot uses internally:

```
record_definition_intent → list_tables / describe_table → draft_spec → validate_spec
```

**Part 1** (no API key): Direct YAML spec parsing — shows what DefinitionBot validates
before minting a `spec_draft_token`.  Each spec type is demonstrated with a valid YAML
string and its parsed attributes.

**Part 2** (requires API key): Two `DefinitionBot.ask()` calls using the ad campaign
dataset — one ratio metric (`avg_cpc`) and one wildcard slice (`country`).  Each run's
trace shows the four mandatory steps and the token-gating in action.

**Part 3** (Anthropic only): Prompt-cache efficiency.  Layer A (workflow rules) and
Layer B (existing catalog) are cached across `ask()` calls.

### Prerequisites

1. Set your Anthropic API key: `export ANTHROPIC_API_KEY=sk-ant-...`
2. Install the agent extra: `pip install aitaem[agent-anthropic]`
3. Set `aitaem_folder_path` below to the path of your local `aitaem` repo.

Set the path to the local folder which contains your AITAEM examples. The following sub-folders are needed inside `aitaem_folder_path`:
- `examples/data/`
- `examples/metrics/`
- `examples/slices/`
- `examples/segments/`

In [ ]:
aitaem_folder_path = "/path/to/aitaem"  # e.g. "/Users/you/Workspace/aitaem"

In [ ]:
import sys
import importlib

sys.path.insert(0, aitaem_folder_path + "/examples")
ex = importlib.import_module("01_definition_bot_example")

## Setup — load specs and connect to DuckDB

In [ ]:
spec_cache, db_path = ex.setup(base_path=aitaem_folder_path)
print(
    f"Catalog: {len(spec_cache.metrics)} metrics, "
    f"{len(spec_cache.slices)} slices, "
    f"{len(spec_cache.segments)} segment(s)"
)
print(f"  Metrics  : {', '.join(spec_cache.metrics)}")
print(f"  Slices   : {', '.join(spec_cache.slices)}")
print(f"  Segments : {', '.join(spec_cache.segments)}")
print(f"DuckDB path ready: {db_path}. Connection will open in Part 2.")

---

## Part 1 — Direct spec parsing: what DefinitionBot validates (no API key)

`MetricSpec.from_yaml()` and `SliceSpec.from_yaml()` are the structural validators
that DefinitionBot's `validate_spec` tool calls internally at check #2.  This part
shows what valid YAML looks like for each spec type and what a structural parse
failure looks like — no LLM involved.

The two new specs we will define in Part 2:
- `avg_cpc` — a ratio metric: `SUM(ad_spend) / SUM(clicks)`
- `country` — a wildcard slice: auto-discovers country values from the column

### Scenario 1 — MetricSpec: `avg_cpc` (ratio metric)

A ratio metric has both `numerator` and `denominator`.  The parser extracts the SQL
expressions and calls `spec.validate()` to find the referenced column names.  These
columns are later checked against the live table schema in `validate_spec` check #5.

In [ ]:
ex.print_spec_parse(
    "MetricSpec: avg_cpc — SUM(ad_spend) / SUM(clicks)",
    """\
metric:
  name: avg_cpc
  description: Average cost per click — total ad spend divided by total clicks
  source: duckdb://ad_campaigns.duckdb/ad_campaigns
  numerator: "SUM(ad_spend)"
  denominator: "SUM(clicks)"
  timestamp_col: date
  format: "currency:USD"
""",
)

### Scenario 2 — SliceSpec (wildcard): `country`

A wildcard slice uses a column name in the `where` field.  At query time the engine
runs `SELECT DISTINCT country FROM …` and groups dynamically.  No hard-coded values
are needed — ideal for high-cardinality dimensions that change over time.

In [ ]:
ex.print_spec_parse(
    "SliceSpec (wildcard): country — auto-discovers values from column",
    """\
slice:
  name: country
  description: Breakdown by country — values auto-discovered from the country column
  where: country
""",
)

### Scenario 3 — Structural parse failure

`numerator` is required for MetricSpec.  Omitting it triggers a `SpecValidationError`
at check #2 of `validate_spec`.  DefinitionBot returns this as a `ValidationIssue` list
so the LLM can fix the draft and call `draft_spec` again with the corrected YAML.

In [ ]:
ex.print_spec_parse(
    "MetricSpec parse failure: missing required 'numerator' field",
    """\
metric:
  name: bad_metric
  source: duckdb://ad_campaigns.duckdb/ad_campaigns
""",
)

---

## Part 2 — DefinitionBot: four-step flow in a real session

Each `ask()` call's trace must show:
1. `record_definition_intent` — LLM records what spec to create and its type
2. `list_tables` + `describe_table` — schema exploration (no column invention)
3. `draft_spec` — stores LLM-written YAML under a `draft_id` (no validation yet)
4. `validate_spec` — runs five checks; mints `spec_draft_token` on full pass

The LLM cannot return a non-null `spec_draft_token` without a successful `validate_spec`
result — the token is the anti-hallucination gate.

In [ ]:
# Close any previous DuckDB connection before re-creating the manager.
# Re-running this cell without closing would hold two locks on the same file.
try:
    conn_mgr.close_all()
except NameError:
    pass  # first run — conn_mgr not yet defined

conn_mgr = ex.ConnectionManager()
conn_mgr.add_connection("duckdb", path=db_path)

bot = ex.DefinitionBot(
    model=ex.MODEL,
    spec_cache=spec_cache,
    connection_manager=conn_mgr,
)

### Ask 1 — ratio metric: `avg_cpc`

The bot must call `list_tables`, then `describe_table` to confirm `ad_spend` and
`clicks` exist before drafting the YAML.  It then calls `validate_spec` which checks
structural validity, name conflict (against existing catalog), and column existence.
On full pass a `spec_draft_token` is minted and returned in `response.payload`.

This call also **warms Layers A and B** in Anthropic's prompt cache.

In [ ]:
response1 = await bot.ask(
    "Define a metric called avg_cpc for average cost per click — "
    "total ad spend divided by total clicks."
)
ex.print_response("Ask 1: avg_cpc — average cost per click", response1)
ex.print_yaml(response1)
ex.print_trace(response1.trace)

### Ask 2 — wildcard slice: `country`

A wildcard slice only needs a column name in `where`.  The bot still calls
`describe_table` to confirm the column exists, then drafts and validates the YAML.
Layer A+B should now be **read from Anthropic's cache**.

In [ ]:
response2 = await bot.ask(
    "Define a wildcard slice called country that auto-discovers "
    "all country values from the country column in the campaigns table."
)
ex.print_response("Ask 2: country — wildcard slice", response2)
ex.print_yaml(response2)
ex.print_trace(response2.trace)

---

## Part 3 — Prompt-cache efficiency (Anthropic only)

DefinitionBot v1 uses the same three-layer system prompt as QueryBot:

| Layer | Content | Cached? |
|-------|---------|--------|
| A | Workflow rules and YAML format reference (identical for all tenants) | Yes |
| B | Per-tenant existing catalog (metric / slice / segment names and details) | Yes |
| C | Today's date (changes each calendar day) | No |

After the first `ask()` warms Layers A+B, subsequent calls should show
`cache_read_tokens > 0` in the usage block.

In [ ]:
if "anthropic:" in bot._model:
    print("Cache efficiency summary (Anthropic only)")
    print("─" * 50)
    ex.cache_summary("Ask 1 (warms cache — no read expected)", response1)
    ex.cache_summary("Ask 2 (Layers A+B should be cached)", response2)
else:
    print(f"Cache check skipped — provider is not Anthropic (model={bot._model!r}).")